In [1]:
import duckdb
import pandas as pd
duckdb.sql("""
           CREATE OR REPLACE TABLE filtered AS
           SELECT message_id, channel_id, user_id, date, text_content, reply_to, n_forwards,
           views, reactions, forward_from, is_vaccine_related, language
           FROM read_json_auto('telegram.jsonl')
           WHERE text_content NOT NULL AND text_content != ''
           AND (language = 'Portuguese' OR language = 'English' OR language = 'Spanish')
           """)

# transform timestamp utc from mili to sec
duckdb.sql("UPDATE filtered SET date = date/1000")

# Limit the forward_from references to just the channels and users that had
# their messages tracked on the original dataset.
duckdb.sql("""
           UPDATE filtered
           SET forward_from = NULL
           WHERE forward_from NOT IN (
                SELECT DISTINCT channel_id
                FROM filtered
                WHERE channel_id IS NOT NULL)
           AND forward_from NOT IN (
                SELECT DISTINCT user_id
                FROM filtered
                WHERE user_id IS NOT NULL)
           """)
duckdb.sql("""
           UPDATE filtered
           SET n_forwards = 0.0
           WHERE forward_from = NULL
           OR reply_to = NULL
           """)

In [2]:
duckdb.sql("""
           SELECT language, COUNT(*) FROM filtered
           GROUP BY language
           """).show()

┌────────────┬──────────────┐
│  language  │ count_star() │
│  varchar   │    int64     │
├────────────┼──────────────┤
│ Portuguese │      2331403 │
│ Spanish    │        71014 │
│ English    │       320762 │
└────────────┴──────────────┘



# Channel table

In [3]:
# the reply_to field only contains references to messages within the same channel.
# because of that, only the forward_from field was considered. 
duckdb.sql("""
           CREATE OR REPLACE VIEW channels AS
           SELECT channel_id, forward_from
           FROM filtered
           WHERE forward_from NOT NULL
           AND forward_from NOT LIKE '<USER%'
           AND channel_id != forward_from
           """)

duckdb.sql("SELECT COUNT(*) FROM channels")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       214780 │
└──────────────┘

In [4]:
# Besides, group in pairs the message channel and the original channel (channel_id and
# forward_from) to compose the future graph edges.

duckdb.sql("""
           CREATE OR REPLACE VIEW channels2 AS
           SELECT channel_id, forward_from, count(*) AS weight
           FROM channels
           WHERE forward_from IN (
                SELECT DISTINCT channel_id
                FROM channels)
           GROUP BY channel_id, forward_from
           """)
duckdb.sql("SELECT COUNT(*) FROM channels2")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         1570 │
└──────────────┘

In [5]:
duckdb.sql("COPY channels2 TO 'channels.csv' (HEADER, DELIMITER ',')")

# Chat table

## Linking forwarding messages
As the forward_from field only specifies the channel from which the message came, it is necessary to search for the orginal forwarded message and link them.

In [11]:
# fwd_messages
duckdb.sql("""
           CREATE OR REPLACE VIEW fwd_messages AS
           SELECT message_id, channel_id, text_content, forward_from
           FROM filtered
           WHERE forward_from NOT NULL
           AND forward_from != ''
           AND forward_from not like '<USER%'
           """)

# the original message will be identified as two messages with same
# text content and one of them referencing the channel of the other.
duckdb.sql("""
           CREATE OR REPLACE TABLE fwd_reply_table AS
           SELECT a.message_id AS message_id, b.message_id AS fwd_reply
           FROM fwd_messages a JOIN filtered b
           ON a.text_content = b.text_content
           AND a.forward_from = b.channel_id
           AND a.message_id != b.message_id
           """)

In [17]:
duckdb.sql("""COPY(
           SELECT message_id, count(*)
           FROM fwd_reply_table
           GROUP BY message_id
           ORDER BY count(*) DESC)
           TO 'temp.csv' (HEADER, DELIMITER ',')
           """)

In [18]:
# after manual inspection, it was seen that all messages that had high
# duplicity rates and were above 93 matches had their content only about
# channel link sharing or channel rules. It was decided to remove those
# links from analysis.
duckdb.sql("""
           DELETE FROM fwd_reply_table
           WHERE message_id IN (
                SELECT message_id
                FROM fwd_reply_table
                GROUP BY message_id
                HAVING COUNT(*) > 92
           )
           """)

print("Forwarded messages:")
print(duckdb.sql("SELECT COUNT(*) FROM fwd_messages"))
print("Total pairs origin-message:")
print(duckdb.sql("SELECT COUNT(*) FROM fwd_reply_table"))
print("Unique pairs origin-message:")
print(duckdb.sql("SELECT COUNT(distinct message_id) FROM fwd_reply_table"))

Forwarded messages:
┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       225704 │
└──────────────┘

Total pairs origin-message:
┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       307156 │
└──────────────┘

Unique pairs origin-message:
┌────────────────────────────┐
│ count(DISTINCT message_id) │
│           int64            │
├────────────────────────────┤
│                     210911 │
└────────────────────────────┘



In [19]:
duckdb.sql("""
           INSERT INTO fwd_reply_table (message_id, fwd_reply)
           SELECT a.message_id, a.reply_to
           FROM filtered AS a
           WHERE a.reply_to IS NOT NULL
           AND a.reply_to != ''
           """)
print("Total forward/reply references:")
print(duckdb.sql("SELECT COUNT(*) FROM fwd_reply_table"))

Total forward/reply references:
┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      1129921 │
└──────────────┘



## Grouping messages into chats

In [20]:
chat_time_period = 180 #seconds

In [21]:
# data from DuckDB into a Pandas DataFrame
df = duckdb.sql("SELECT * FROM filtered").df()
fwd_reply_df = duckdb.sql("SELECT * FROM fwd_reply_table").df()

df = df.sort_values(['channel_id', 'date'])

# time difference between consecutive messages in the same channel
# .diff() gives the difference with the previous row
df['time_gap'] = df.groupby('channel_id')['date'].diff()

# a new chat starts if time_gap > time OR if it's the first message in the channel (NaN)
df['is_new_chat'] = (df['time_gap'] > chat_time_period) | (df['time_gap'].isna())
df['chat_counter'] = df.groupby('channel_id')['is_new_chat'].cumsum().astype(int)
df['chat_id'] = df['channel_id'].astype(str) + "_" + df['chat_counter'].astype(str)

msg_to_chat_map = df.set_index('message_id')['chat_id'].to_dict()
fwd_reply_df['source_chat'] = fwd_reply_df['message_id'].map(msg_to_chat_map)
fwd_reply_df['target_chat'] = fwd_reply_df['fwd_reply'].map(msg_to_chat_map)

# drop null and self references
links = fwd_reply_df.dropna(subset=['source_chat', 'target_chat'])
links = links[links['source_chat'] != links['target_chat']]

chat_links = (
    links.groupby('source_chat')['target_chat']
    .apply(lambda x: list(set(x)))
    .reset_index()
    .rename(columns={'source_chat': 'chat_id', 'target_chat': 'linked_chats'})
)

# Aggregate message-level data into chat-level data
chat_df = df.groupby('chat_id').agg({
    'channel_id': 'first',
    'text_content': lambda x: "\n".join(x.astype(str)),
    'user_id': lambda x: list(x.unique()),
    'views': 'mean',
    'reactions': 'sum',
    'n_forwards': 'sum',
    'date': ['min', 'max'] 
}).reset_index()

chat_df.columns = [
    'chat_id', 'channel_id', 'text_content', 'participants', 
    'avg_views', 'total_reactions', 'total_forwards', 'start_time', 'end_time'
]

# Merge the links into the chat_df
chat_df = chat_df.merge(chat_links, on='chat_id', how='left')

# Ensure 'linked_chats' is an empty list instead of NaN for chats with no outgoing links
chat_df['linked_chats'] = chat_df['linked_chats'].apply(lambda d: d if isinstance(d, list) else [])

# Optional: Calculate the duration of the chat in seconds
chat_df['duration_seconds'] = chat_df['end_time'] - chat_df['start_time']

In [22]:
chat_df.to_parquet('telegram_chats_2020_2025.parquet', index=False)

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.

In [23]:
pip install pyarrow fastparquet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 2.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 2.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 2.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.6/202.6 kB 2.5 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [24]:
# 1. General Overview
print(f"Total Chats Created: {len(chat_df):,}")
print(f"Total Unique Channels: {chat_df['channel_id'].nunique()}")

# 2. Distribution of Chat Sizes
chat_df['num_participants'] = chat_df['participants'].apply(len)
chat_df['num_links'] = chat_df['linked_chats'].apply(len)

print("\n--- Structural Statistics ---")
stats = chat_df[['num_participants', 'num_links', 'avg_views', 'duration_seconds']].describe()
print(stats)

# 3. Network Health
chats_with_links = (chat_df['num_links'] > 0).sum()
pct_linked = (chats_with_links / len(chat_df)) * 100
print(f"\nChats that reference others: {chats_with_links:,} ({pct_linked:.2f}%)")

# 4. Top 'Viral' Chats (by total forwards)
print("\n--- Top 5 Most Forwarded Chats ---")
print(chat_df.nlargest(5, 'total_forwards')[['chat_id', 'total_forwards', 'num_links']])

Total Chats Created: 1,235,164
Total Unique Channels: 119

--- Structural Statistics ---
       num_participants     num_links     avg_views  duration_seconds
count      1.235164e+06  1.235164e+06  9.060290e+05      1.235164e+06
mean       1.309723e+00  4.177130e-01  8.296992e+03      6.724776e+01
std        3.723272e+00  1.779301e+00  4.908177e+04      2.753048e+02
min        1.000000e+00  0.000000e+00  0.000000e+00      0.000000e+00
25%        1.000000e+00  0.000000e+00  7.630000e+02      0.000000e+00
50%        1.000000e+00  0.000000e+00  1.876000e+03      0.000000e+00
75%        1.000000e+00  0.000000e+00  4.490500e+03      4.100000e+01
max        1.011000e+03  2.190000e+02  2.108251e+07      3.809100e+04

Chats that reference others: 296,166 (23.98%)

--- Top 5 Most Forwarded Chats ---
                                          chat_id  total_forwards  num_links
77383   <CHANNEL_HASH:15b89df7bcd31a345556>_24460         69213.0          0
611571  <CHANNEL_HASH:5e16f210a7dc1d2b7ff2>_